# Golden Sets

**Goal:** Build a golden set for the RAG system from section 02, and run cheap deterministic checks over it.

Part of [ai-engineer-notebooks](https://github.com/calmrocks/ai-engineer-notebooks) — the hands-on companion to the [FDE / AI Engineer transition plan](https://www.calm.rocks/resources/career-development/transition-fde-ai-engineer/).


In [ ]:
%pip install -q groq sentence-transformers numpy

In [ ]:
import os

# In Colab, read the key from Secrets (key icon in the left sidebar).
# Locally, set the GROQ_API_KEY env var instead.
try:
    from google.colab import userdata
    os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')
except ImportError:
    assert os.environ.get('GROQ_API_KEY'), 'Set GROQ_API_KEY'

from groq import Groq
client = Groq()
MODEL = 'openai/gpt-oss-120b'  # update if you get a 404 — see 00-setup/00-environment.ipynb to list available models

## Why this notebook exists

Evals are the single strongest "has actually shipped" signal an AI engineer can send. A demo shows the system works once; an eval shows you know when it doesn't. Anyone can wire retrieval to a model and get a good answer on stage. The engineer who gets hired is the one who can say "we're at 87% on our golden set, the three failures are all multi-hop questions, and here's the diff from last week."

This is also the exact instinct you already have from backend work: every production bug becomes a regression test. Same move here — every bad answer becomes a golden-set case.

In this notebook we build the golden set and run two deterministic metrics over it. The next two notebooks add LLM judging and a regression harness on top.


## The system under test

The next two cells rebuild the RAG system from section 02 in compact form: download and clean ten IETF RFCs, chunk them by section, embed with a small local model, retrieve by cosine similarity, and answer with citations.

This is a deliberate copy, not an import. Each notebook has to run top-to-bottom in a fresh Colab runtime, and self-containment beats DRY for teaching material. If the details are unfamiliar, work through `02-rag/` first — here they're just the thing we're evaluating.


In [ ]:
# Compact copy of the 02-rag pipeline. Self-containment over DRY: this notebook must run
# standalone in Colab. See 02-rag/ for the full walkthrough of every design decision here.
import re
import urllib.request

RFCS = [791, 793, 1035, 2616, 4271, 5321, 6455, 6749, 7540, 9110]
os.makedirs('data/rfc', exist_ok=True)
for n in RFCS:
    path = f'data/rfc/rfc{n}.txt'
    if not os.path.exists(path):  # skip if cached from an earlier notebook
        urllib.request.urlretrieve(f'https://www.rfc-editor.org/rfc/rfc{n}.txt', path)

def clean_rfc(text):
    """Strip form feeds and the page header/footer lines RFC txt files carry."""
    text = text.replace('\f', '\n')
    lines = [l for l in text.split('\n')
             if not re.search(r'\[Page \d+\]\s*$', l)          # footers
             and not re.match(r'^\s*RFC \d+.*\d{4}\s*$', l)]   # headers
    return re.sub(r'\n{3,}', '\n\n', '\n'.join(lines))

def chunk_rfc(text, rfc, max_chars=2000):
    """Section-aware chunking: split on numbered headings, then cap chunk size."""
    parts = re.split(r'\n(?=\d+(?:\.\d+)*\.?\s+[A-Z])', text)
    chunks = []
    for part in parts:
        part = part.strip()
        while len(part) > max_chars:
            cut = part.rfind('\n\n', 0, max_chars)
            cut = cut if cut > 200 else max_chars
            chunks.append({'rfc': rfc, 'text': part[:cut].strip()})
            part = part[cut:].strip()
        if len(part) > 100:
            chunks.append({'rfc': rfc, 'text': part})
    return chunks

chunks = []
for n in RFCS:
    with open(f'data/rfc/rfc{n}.txt') as f:
        chunks += chunk_rfc(clean_rfc(f.read()), n)
print(f'{len(chunks)} chunks from {len(RFCS)} RFCs')


In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer('all-MiniLM-L6-v2')
emb = embedder.encode([c['text'] for c in chunks],
                      normalize_embeddings=True, show_progress_bar=True)

def retrieve(query, k=5):
    q = embedder.encode([query], normalize_embeddings=True)[0]
    scores = emb @ q  # cosine similarity: vectors are unit-normalized
    return [chunks[i] for i in np.argsort(-scores)[:k]]

def answer_question(question, k=5):
    """Retrieve top-k chunks, answer with citations. Returns (answer, hits)."""
    hits = retrieve(question, k)
    context = '\n\n'.join(f"[RFC {h['rfc']}]\n{h['text']}" for h in hits)
    resp = client.chat.completions.create(
        model=MODEL, max_tokens=1024,
        messages=[
            {'role': 'system',
             'content': ('Answer using ONLY the provided RFC excerpts. Cite the RFC number for '
                         'each claim, e.g. (RFC 9110). If the excerpts do not contain the answer, '
                         'say plainly that the corpus does not cover it. Do not guess.')},
            {'role': 'user', 'content': f'{context}\n\nQuestion: {question}'},
        ],
    )
    return resp.choices[0].message.content, hits

## What makes a golden set good

A golden set is a list of question/reference pairs you trust. Four properties matter more than size:

- **It covers the failure modes you fear, not just the happy path.** Ours mixes easy lookups (404), multi-part answers (OAuth roles), and questions whose keywords appear in several RFCs (idempotent methods live in both RFC 2616 and RFC 9110 — which one does the retriever pick?).
- **It includes unanswerable questions.** The corpus has no QUIC, no TLS, no IMAP. The correct answer to those questions is a refusal. A RAG system that confidently answers them is hallucinating, and you want a test that catches it.
- **It's small enough to run on every change.** ~15 cases is a few cents and a couple of minutes. A 500-case suite you run monthly catches regressions a month late.
- **It grows from production failures.** When a user gets a bad answer, that question goes in the set with the answer they should have gotten. Every bug becomes a test case — the same discipline as regression tests, applied to model behavior.

Each case below carries `must_mention` (keywords a correct answer can't omit) and `source_rfc` (the document the retriever must surface). Those two fields are what make cheap deterministic checking possible.


In [ ]:
# The golden set: ~15 cases written against the actual corpus. Each case pins down
# what a correct answer must contain (must_mention) and which RFC the retriever
# should surface (source_rfc). Unanswerable cases have source_rfc=None -- the correct
# behavior there is to decline, and hallucinating an answer is a hard failure.
GOLDEN_SET = [
    {'question': 'How does the TCP three-way handshake establish a connection?',
     'reference_answer': 'The initiator sends a SYN carrying its initial sequence number; '
                         'the peer responds with SYN-ACK carrying its own sequence number and '
                         'acknowledging the first; the initiator replies with an ACK. Both '
                         'sides now have synchronized sequence numbers.',
     'source_rfc': 793, 'must_mention': ['SYN', 'ACK', 'sequence'], 'unanswerable': False},
    {'question': 'What does the HTTP 404 status code mean?',
     'reference_answer': '404 Not Found: the origin server did not find a current '
                         'representation for the target resource, or is not willing to '
                         'disclose that one exists.',
     'source_rfc': 9110, 'must_mention': ['404', 'not found'], 'unanswerable': False},
    {'question': 'What are the four roles defined in the OAuth 2.0 framework?',
     'reference_answer': 'Resource owner, resource server, client, and authorization server.',
     'source_rfc': 6749,
     'must_mention': ['resource owner', 'client', 'authorization server', 'resource server'],
     'unanswerable': False},
    {'question': 'What is the difference between an A record and a CNAME record in DNS?',
     'reference_answer': 'An A record maps a name to a 32-bit IPv4 host address; a CNAME '
                         'record maps an alias to the canonical name of its target.',
     'source_rfc': 1035, 'must_mention': ['address', 'canonical'], 'unanswerable': False},
    {'question': 'How does a client upgrade an HTTP connection to a WebSocket connection?',
     'reference_answer': 'The client sends a GET request with Upgrade: websocket and '
                         'Connection: Upgrade headers plus a Sec-WebSocket-Key; the server '
                         'replies 101 Switching Protocols with a Sec-WebSocket-Accept value '
                         'derived from the key.',
     'source_rfc': 6455, 'must_mention': ['Upgrade', '101', 'Sec-WebSocket-Key'],
     'unanswerable': False},
    {'question': 'Which SMTP commands send a mail message, and in what order?',
     'reference_answer': 'MAIL FROM identifies the sender, one or more RCPT TO commands name '
                         'recipients, then DATA introduces the message text, terminated by a '
                         'line containing only a period.',
     'source_rfc': 5321, 'must_mention': ['MAIL', 'RCPT', 'DATA'], 'unanswerable': False},
    {'question': 'What is the purpose of the Time to Live field in the IPv4 header?',
     'reference_answer': 'TTL is an upper bound on datagram lifetime. Every module that '
                         'processes the datagram decrements it, and the datagram is discarded '
                         'when it reaches zero, so packets cannot loop forever.',
     'source_rfc': 791, 'must_mention': ['zero', 'discard'], 'unanswerable': False},
    {'question': 'What is the AS_PATH attribute in BGP and why does it matter?',
     'reference_answer': 'AS_PATH records the sequence of autonomous systems a route '
                         'advertisement has traversed. BGP uses it for loop detection (a '
                         'router rejects routes containing its own AS number) and in route '
                         'selection.',
     'source_rfc': 4271, 'must_mention': ['autonomous system', 'loop'], 'unanswerable': False},
    {'question': 'How does HTTP/2 carry multiple requests over a single connection?',
     'reference_answer': 'HTTP/2 multiplexes exchanges as independent streams over one '
                         'connection. Each frame carries a stream identifier, so concurrent '
                         'requests and responses interleave without blocking each other.',
     'source_rfc': 7540, 'must_mention': ['stream', 'frame'], 'unanswerable': False},
    {'question': 'Which HTTP methods are defined as idempotent?',
     'reference_answer': 'PUT and DELETE, plus all safe methods: GET, HEAD, OPTIONS, TRACE.',
     'source_rfc': 9110, 'must_mention': ['PUT', 'DELETE', 'GET'], 'unanswerable': False},
    {'question': 'Why does a TCP connection enter the TIME-WAIT state, and for how long?',
     'reference_answer': 'The side that closes actively waits in TIME-WAIT for twice the '
                         'maximum segment lifetime (2*MSL), so delayed segments from the old '
                         'incarnation die off before the socket pair can be reused.',
     'source_rfc': 793, 'must_mention': ['TIME-WAIT', 'MSL'], 'unanswerable': False},
    {'question': 'Walk through the OAuth 2.0 authorization code grant flow.',
     'reference_answer': 'The client redirects the resource owner to the authorization '
                         'server; after authentication and consent the server redirects back '
                         'with an authorization code; the client exchanges the code, with its '
                         'own credentials, at the token endpoint for an access token.',
     'source_rfc': 6749, 'must_mention': ['authorization code', 'access token', 'redirect'],
     'unanswerable': False},
    {'question': 'What does HTTP status 503 indicate, and which header can accompany it?',
     'reference_answer': '503 Service Unavailable: the server is currently unable to handle '
                         'the request, due to overload or maintenance. It may send Retry-After '
                         'to indicate how long to wait.',
     'source_rfc': 9110, 'must_mention': ['503', 'Retry-After'], 'unanswerable': False},
    # --- deliberately unanswerable: correct behavior is to decline ---
    {'question': 'How does QUIC combine the transport and cryptographic handshakes?',
     'reference_answer': 'Declines: QUIC (RFC 9000) is not in the corpus.',
     'source_rfc': None, 'must_mention': [], 'unanswerable': True},
    {'question': 'What cipher suites does TLS 1.3 define?',
     'reference_answer': 'Declines: TLS 1.3 (RFC 8446) is not in the corpus.',
     'source_rfc': None, 'must_mention': [], 'unanswerable': True},
    {'question': 'Which port does IMAP use, and how does a client select a mailbox?',
     'reference_answer': 'Declines: IMAP is not in the corpus (SMTP is the only mail '
                         'protocol here).',
     'source_rfc': None, 'must_mention': [], 'unanswerable': True},
]
print(len(GOLDEN_SET), 'cases,',
      sum(c['unanswerable'] for c in GOLDEN_SET), 'unanswerable')


## Two deterministic metrics, before any LLM judging

Before spending money on a judge model, extract everything you can from checks that are free and instant:

1. **Retrieval hit-rate** — is the right RFC anywhere in the top-k? This isolates the retriever: if the right document never showed up, the generator never had a chance, and no amount of prompt tuning will fix it.
2. **Keyword coverage** — what fraction of `must_mention` terms appear in the generated answer? An answer about the TCP handshake that never says "SYN" is wrong, full stop, and you don't need a judge to know it.

For the unanswerable cases we do a crude marker scan for decline language. It's brittle — that's fine. Deterministic checks are brittle but free and fast; their job is to catch gross regressions and triage what's worth sending to the expensive judge. Cheap filters first, expensive scoring second — deliberately the same funnel logic as retrieval→reranking in section 02.


In [ ]:
def retrieval_hit(case, hits):
    """Was the right RFC anywhere in the retrieved top-k? None for unanswerable cases."""
    if case['source_rfc'] is None:
        return None
    return case['source_rfc'] in [h['rfc'] for h in hits]

def keyword_coverage(case, answer):
    """Fraction of must_mention keywords present in the answer (case-insensitive)."""
    if not case['must_mention']:
        return None
    a = answer.lower()
    return sum(kw.lower() in a for kw in case['must_mention']) / len(case['must_mention'])

DECLINE_MARKERS = ['does not cover', "doesn't cover", 'not covered', 'do not contain',
                   'does not contain', 'not in the corpus', 'cannot answer',
                   'no information', 'not addressed', 'do not include']

def declined(answer):
    """Crude decline detector for the unanswerable cases. Brittle by design -- the
    LLM judge in the next notebook does this properly. Here it just has to catch
    the gross failure: confidently answering a question the corpus can't support."""
    a = answer.lower()
    return any(m in a for m in DECLINE_MARKERS)


In [ ]:
# Run the whole golden set through the RAG system.
# Cost note: 16 calls to MODEL (sonnet) -- one per case. Small enough to run on
# every change, which is the whole point of keeping the set small.
results = []
for case in GOLDEN_SET:
    answer, hits = answer_question(case['question'])
    row = {
        'question': case['question'],
        'answer': answer,
        'hit': retrieval_hit(case, hits),
        'coverage': keyword_coverage(case, answer),
        'declined': declined(answer),
        'unanswerable': case['unanswerable'],
    }
    if case['unanswerable']:
        row['ok'] = row['declined']  # only correct behavior is to decline
    else:
        row['ok'] = bool(row['hit']) and row['coverage'] >= 0.6
    results.append(row)
    print('PASS' if row['ok'] else 'FAIL', '|', case['question'][:65])


In [ ]:
# Results table -- plain Python, no pandas needed for 16 rows.
def fmt(v):
    if v is None: return '   -'
    if isinstance(v, bool): return ' yes' if v else '  NO'
    return f'{v:.2f}'

print(f"{'hit':>4}  {'cov':>4}  {'ok':>4}  question")
print('-' * 78)
for r in results:
    print(f"{fmt(r['hit'])}  {fmt(r['coverage'])}  {fmt(r['ok'])}  "
          f"{r['question'][:60]}")

answerable = [r for r in results if not r['unanswerable']]
unanswerable = [r for r in results if r['unanswerable']]
print('-' * 78)
print(f'retrieval hit-rate : {sum(r["hit"] for r in answerable)}/{len(answerable)}')
print(f'mean kw coverage   : '
      f'{sum(r["coverage"] for r in answerable) / len(answerable):.2f}')
print(f'unanswerable OK    : {sum(r["ok"] for r in unanswerable)}/{len(unanswerable)}')


Run this and note where the failures cluster. Typical pattern on this corpus: hit-rate misses land on questions whose vocabulary spans multiple RFCs (HTTP questions retrieving RFC 2616 instead of 9110 is common — both describe status codes, and cosine similarity can't tell "current spec" from "obsoleted spec"). Coverage misses are often near-misses: the answer is right but phrases a keyword differently, which is precisely the brittleness that motivates the LLM judge in the next notebook.

Also inspect an unanswerable failure if you get one — read the actual answer. A confident, well-cited, wrong answer is the scariest failure mode a RAG system has, and now you have a test that catches it.

**Where this sits in the stack:** deterministic checks (this notebook) → LLM judge for the semantic layer (next notebook) → a regression harness that diffs runs and gates changes (notebook 3). Hosted eval platforms and OSS frameworks (promptfoo, Braintrust, LangSmith and friends) package this same layering with dashboards, run history, and CI hooks — worth adopting later, but build it by hand once so you know what they're automating.


## Exercises

1. **Add five hard cases.** Write cases you expect the system to *fail*: a question answered in a table or ASCII diagram (RFC 793's state diagram), a question whose answer spans two RFCs, and a question using synonyms that never appear in the text ("web address" for URI). Run the set and check your predictions.
2. **Fix the decline detector's blind spot.** Craft an unanswerable question where the model declines using words not in `DECLINE_MARKERS`, so `declined()` reports a false failure. Then invert the check: instead of detecting declines, detect *confident answers* (citations present, no hedging) and flag those. Which direction is less brittle?
3. **Measure hit-rate sensitivity to k.** Rerun the retrieval-only metric (no generation calls needed — just `retrieve()`) for k = 1, 3, 5, 10 and print hit-rate per k. Find the smallest k that keeps hit-rate at its maximum.
4. **Turn a "production bug" into a case.** Ask the system three off-set questions until you find a bad answer. Write the golden-set entry for it — question, reference answer from the RFC text, `must_mention`, `source_rfc` — and confirm the new case fails before any fix and passes after you raise k or improve chunking.
